In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
# import pyfredapi as pf
import stockstats

# from time import sleep

# from fred_api_key import FRED_API_KEY
from fred_macro import FRED_MACRO_TICKERS
from index_ticker import SP500, US_INDEX_TICKERS, MACRO_TICKERS, INT_TICKERS
from stockstats_technicals import STOCKSTATS_TECHNICALS

# API_KEY = FRED_API_KEY

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [2]:
# Read indices_from_2000.csv from cache
df_stock = pd.read_csv(CACHE_PATH / "indices_from_2000.csv", header=[0,1], index_col=0)

In [3]:
# Read FOMC meeting information from cache
fomc = pd.read_csv(CACHE_PATH / "fomc_calendar_2000_present.csv")

In [4]:
# CPI derivative -> Inflation rate (2% - Golden number)

macro = pd.read_csv(CACHE_PATH / "macro_data.csv")

In [5]:
# Ensure datetime
df_stock = df_stock.copy()
df_stock.index = pd.to_datetime(df_stock.index)

fomc = fomc.copy()
fomc["date"] = pd.to_datetime(fomc["date"])

# Keep only announcement (decision) dates
fomc_announcements = (
    fomc.loc[fomc["is_fomc_day"] == 1, ["date"]]
    .drop_duplicates()
    .rename(columns={"date": "last_fomc_date"})
    .sort_values("last_fomc_date")
)

# Trading dates
tmp = (
    pd.DataFrame({"date": df_stock.index})
    .sort_values("date")
)

# Match each trading day to the most recent FOMC announcement
tmp = pd.merge_asof(
    tmp,
    fomc_announcements,
    left_on="date",
    right_on="last_fomc_date",
    direction="backward",
)

# Calendar days since last FOMC announcement
tmp["days_since_fomc"] = (
    tmp["date"] - tmp["last_fomc_date"]
).dt.days

# Add to df_stock
df_stock[("days_since_fomc", "")] = (
    tmp.set_index("date")["days_since_fomc"]
)

In [6]:
# Join macro data
macro.columns = pd.MultiIndex.from_product( # To match multi-index used by df_stock
    [["Macro"], macro.columns],
    names=df_stock.columns.names,
)

df = df_stock.join(macro, how="left")

In [7]:
sp = pd.DataFrame({
    "open": df[("Open", SP500)],
    "high": df[("High", SP500)],
    "low": df[("Low", SP500)],
    "close": df[("Close", SP500)],
    "volume": df[("Volume", SP500)],
})

sp = stockstats.StockDataFrame.retype(sp)

technical_features = []

for feature in STOCKSTATS_TECHNICALS:
    try:
        df[("Technical", feature)] = sp[feature]
        technical_features.append(feature)
    except Exception as e:
        print(f"Skipping {feature}: {e}")

# print(sp.head())
sp.to_csv(CACHE_PATH / "sp500_technicals.csv")

1. Add a few derived macro features.
2. Construct the target.
3. Remove columns that should not be predictors.
4. Remove rows with unavailable features.
5. Fit a regularized baseline using an expanding time-series split.

In [34]:
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Yield spread (10Y - 2Y)
if ("Macro", "DGS10") in df.columns and ("Macro", "DGS2") in df.columns:
    df[("Macro", "YieldSpread")] = (
        df[("Macro", "DGS10")]
        - df[("Macro", "DGS2")]
    )

# Real Fed Funds Rate
if (
    ("Macro", "DFF") in df.columns
    and ("Macro", "CORESTICKM159SFRBATL") in df.columns
):
    df[("Macro", "RealFedFunds")] = (
        df[("Macro", "DFF")]
        - df[("Macro", "CORESTICKM159SFRBATL")]
    )

# VVIX / VIX ratio
if (
    ("Close", "^VVIX") in df.columns
    and ("Close", "^VIX") in df.columns
):
    df[("Macro", "VVIX_VIX")] = (
        df[("Close", "^VVIX")]
        / df[("Close", "^VIX")]
    )


TARGET = ("Ret_1", SP500)

START_DATE = "2001-01-01"
y = df.loc[df.index >= pd.Timestamp(START_DATE), TARGET].shift(-1)

remove_first_level = {
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
}

X = df.loc[
    :,
    ~df.columns.get_level_values(0).isin(remove_first_level)
]

# X = X.drop(columns=[("Ret_1", SP500)])
X.columns = [
    f"{level0}_{level1}".rstrip("_")
    for level0, level1 in X.columns
]

X = X.dropna(axis=1, how="all")

Drop VVIX-derived features to train on the full sample.

In [35]:
vvix_cols = [
    c for c in X.columns
    if "VVIX" in c
]

X = X.drop(columns=vvix_cols)

Combine predictors and target. Recover X and y.

In [36]:
dataset = pd.concat(
    [X, y.rename("target")],
    axis=1,
)

dataset = dataset.dropna()

X = dataset.drop(columns="target")
y = dataset["target"]

Fit an Elastic Net/Ridge baseline using expanding-window cross-validation.

In [ ]:
tscv = TimeSeriesSplit(
    n_splits=5,
)

pipeline = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "model",
            Ridge(
                alpha=100.0,
                # l1_ratio=0.5,
                max_iter=10000,
                random_state=42,
            ),
        ),
    ]
)

Evaluate:

In [53]:
oos_pred = pd.Series(index=y.index, dtype=float)

for train_idx, test_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    pipeline.fit(X_train, y_train)

    oos_pred.iloc[test_idx] = pipeline.predict(X_test)

Compute metrics:

In [54]:
mask = oos_pred.notna()

rmse = np.sqrt(
    mean_squared_error(
        y[mask],
        oos_pred[mask],
    )
)

r2 = r2_score(
    y[mask],
    oos_pred[mask],
)

corr = np.corrcoef(
    y[mask],
    oos_pred[mask],
)[0, 1]

print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")
print(f"Corr : {corr:.6f}")

RMSE : 0.012216
R²   : -0.278242
Corr : 0.075389


In [ ]:
coef = pd.Series(
    pipeline.named_steps["model"].coef_,
    index=X.columns,
)

coef = coef[coef != 0]

print(len(coef))
sorted = coef.sort_values(key=np.abs)
print(sorted.tail(20))
# sorted.to_csv(CACHE_PATH / "coef.csv")


240
Ret_20_^GSPTSE           -0.001194
Ret_1_^RUT                0.001208
Ret_60_^GSPC             -0.001247
Technical_stochrsi_14     0.001421
Technical_kdjj           -0.001435
Technical_close_75_mad    0.001454
Ret_120_^GDAXI           -0.001470
Ret_5_^STOXX50E          -0.001577
Technical_ppo            -0.001597
Technical_close_75_z      0.001638
Ret_60_^TWII              0.001727
Vol_5_^STI                0.001749
Ret_1_^DJI               -0.001751
Ret_5_^GSPC              -0.001813
Vol_10_^RUT               0.001902
Ret_1_^GSPC              -0.001945
Vol_5_^GDAXI              0.001976
Ret_5_^GDAXI              0.002019
Ret_1_^NYA                0.002158
Ret_1_^GSPTSE            -0.003089
dtype: float64


In [56]:
print(y.describe())
print(oos_pred.describe())

count    4721.000000
mean        0.000415
std         0.012540
min        -0.119841
25%        -0.004187
50%         0.000720
75%         0.005967
max         0.115800
Name: target, dtype: float64
count    3930.000000
mean       -0.001521
std         0.006199
min        -0.032130
25%        -0.004562
50%        -0.001290
75%         0.001860
max         0.051150
dtype: float64


In [57]:
print(np.max(np.abs(y)))
print(np.max(np.abs(oos_pred)))

0.1198405524039344
0.0511496445472985
